# Frozen Quality-Matched Nuclear-IOD Genome-Size Analysis

            This notebook analyzes every nucleus in the final reviewed panel:
            721 manually accepted nuclei from 20 species. The primary estimator
            is the mean of the image-specific median IOD values, giving every
            observed image/specimen equal weight.

            **Interpretation boundary:** this is a relative nuclear-IOD
            genome-size proxy. It is not an absolute genome-size estimate,
            C-value, or picogram measurement because no independent
            DNA-content standard was imaged and calibrated with these samples.


## Exact frozen inputs and analysis setup


In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display, HTML


def find_project_root(start=None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "paths.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "path_analysis" / "scripts"))
import build_frozen_genome_iod_notebook as analysis

FROZEN_PATH = analysis.FROZEN_PATH
MATCH_MANIFEST_PATH = analysis.MATCH_MANIFEST_PATH
REPORT_DIR = analysis.REPORT_DIR
FIGURE_DIR = analysis.FIGURE_DIR
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

frozen = pd.read_csv(FROZEN_PATH, low_memory=False)
match_manifest = json.loads(MATCH_MANIFEST_PATH.read_text())
species_summary, image_summary = analysis.summarize_species(
    frozen, n_bootstrap=2000, seed=20260710
)
quality_balance, quality_residual = analysis.frozen_quality_diagnostics(frozen)
anchor = float(species_summary["relative_iod_anchor"].iloc[0])

plt.rcParams.update({
    "figure.dpi": 115,
    "savefig.dpi": 190,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 800)
pd.set_option("display.width", 180)


def short_species(value):
    return str(value).replace("D. ", "")


def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    return path


print("Frozen source:", FROZEN_PATH)
print("Frozen SHA-256:", analysis.sha256_file(FROZEN_PATH))
print("Rows:", len(frozen), "| Species:", frozen["species"].nunique())
print("Relative-index anchor IOD:", round(anchor, 3))


## Freeze and source audit

            These checks are hard failures: all rows must be reviewed keeps,
            no rejected key can appear, every species must retain at least 30
            nuclei, and the source manifest must mark the panel frozen.


In [ ]:
decisions = analysis.load_latest_decisions()
problem_keys = set(map(
    tuple,
    decisions.loc[decisions["decision"].eq("problem"), ["species", "review_key"]]
    .itertuples(index=False, name=None),
))
frozen_keys = set(map(
    tuple,
    frozen[["species", "review_key"]].itertuples(index=False, name=None),
))
counts = frozen.groupby("species").size()
audit = pd.DataFrame({
    "check": [
        "manifest freeze status",
        "reviewed nuclei",
        "primary species",
        "images/specimens",
        "minimum nuclei per species",
        "maximum nuclei per species",
        "duplicate keys",
        "rejected-key leaks",
        "all rows explicitly reviewed keep",
    ],
    "value": [
        match_manifest["freeze_status"],
        len(frozen),
        frozen["species"].nunique(),
        f'{frozen["filename"].nunique()} / {frozen["specimen_group"].nunique()}',
        int(counts.min()),
        int(counts.max()),
        int(frozen.duplicated(["species", "review_key"]).sum()),
        len(frozen_keys & problem_keys),
        bool(frozen["review_status"].eq("reviewed_keep").all()),
    ],
})
display(audit)
assert match_manifest["freeze_status"] == "frozen_reviewed_only_no_further_replacement"
assert frozen.duplicated(["species", "review_key"]).sum() == 0
assert not (frozen_keys & problem_keys)
assert frozen["review_status"].eq("reviewed_keep").all()
assert counts.min() >= 30


## Complete species-level results


In [ ]:
summary_columns = [
    "relative_iod_rank",
    "species",
    "n_nuclei",
    "n_images",
    "n_specimens",
    "iod_equal_image_estimate",
    "iod_equal_image_ci_low",
    "iod_equal_image_ci_high",
    "relative_iod_index",
    "relative_iod_ci_low",
    "relative_iod_ci_high",
    "iod_pooled_nucleus_median",
    "iod_trimmed_mean_10pct",
    "pooled_median_difference_pct",
    "trimmed_mean_difference_pct",
    "median_nucleus_area_um2",
    "median_nucleus_mean_od",
    "image_median_iod_min",
    "image_median_iod_max",
    "estimate_support",
]
display(
    species_summary[summary_columns]
    .style.format({
        "iod_equal_image_estimate": "{:.2f}",
        "iod_equal_image_ci_low": "{:.2f}",
        "iod_equal_image_ci_high": "{:.2f}",
        "relative_iod_index": "{:.3f}",
        "relative_iod_ci_low": "{:.3f}",
        "relative_iod_ci_high": "{:.3f}",
        "iod_pooled_nucleus_median": "{:.2f}",
        "iod_trimmed_mean_10pct": "{:.2f}",
        "pooled_median_difference_pct": "{:+.1f}%",
        "trimmed_mean_difference_pct": "{:+.1f}%",
        "median_nucleus_area_um2": "{:.2f}",
        "median_nucleus_mean_od": "{:.4f}",
        "image_median_iod_min": "{:.2f}",
        "image_median_iod_max": "{:.2f}",
    })
)


## All 721 raw nuclei

            Every accepted nucleus is included below. IOD, its two algebraic
            components, image identity, and the two matching-quality features
            are visible. The frozen source download retains all 189 provenance
            and QC columns.


In [ ]:
raw_columns = [
    "species",
    "filename",
    "specimen_group",
    "nucleus_label",
    "review_key",
    "nuc_iod",
    "nuc_area_um2",
    "nuc_mean_od",
    "match_log_edge_sharpness",
    "match_log_relative_ring_noise",
    "quality_match_distance",
    "review_status",
]
raw_display = frozen[raw_columns].sort_values(
    ["species", "filename", "nuc_iod"], kind="mergesort"
)
relative_source_link = (
    "../../data/external/derived/image_quality_matched_genome_iod/"
    "image_quality_matched_nuclei_frozen_reviewed.csv.gz"
)
display(HTML(
    f'<p><a href="{relative_source_link}">Download the complete frozen 189-column CSV</a></p>'
    '<div style="max-height:620px;overflow:auto;border:1px solid #ccd3da;padding:4px">'
    + raw_display.to_html(index=False, float_format=lambda value: f"{value:.5f}")
    + "</div>"
))
print("Displayed raw nuclei:", len(raw_display))


## Raw nucleus-level IOD distributions


In [ ]:
order = species_summary.sort_values("iod_equal_image_estimate")["species"].tolist()
rng = np.random.default_rng(20260710)
fig, ax = plt.subplots(figsize=(13, 12))
for y, species in enumerate(order):
    values = frozen.loc[frozen["species"].eq(species), "nuc_iod"].to_numpy()
    jitter = rng.normal(0, 0.065, size=len(values))
    ax.scatter(values, y + jitter, s=17, alpha=0.52, color="#426B8A", edgecolor="none")
    estimate = species_summary.loc[
        species_summary["species"].eq(species), "iod_equal_image_estimate"
    ].iloc[0]
    ax.scatter(estimate, y, marker="D", s=48, color="#B33A3A", zorder=5)
ax.set_yticks(range(len(order)), [short_species(value) for value in order])
ax.set_xlabel("Nuclear IOD (raw image units)")
ax.set_ylabel("Species")
ax.set_title("Every reviewed nucleus; red diamond = equal-image species estimate")
ax.grid(axis="x", alpha=0.18)
save_figure(fig, "01_all_raw_nuclear_iod.png")
plt.show()


## Primary relative IOD estimates and conditional uncertainty

            The species index is normalized so the median species estimate is
            1.0. Confidence intervals use hierarchical resampling of images and
            then nuclei within images. They are conditional on the images and
            specimens observed here; the three single-image species cannot
            express between-image uncertainty. The index intervals divide the
            raw-IOD intervals by the observed anchor and therefore treat that
            anchor as fixed.


In [ ]:
plot_data = species_summary.sort_values("relative_iod_index")
y = np.arange(len(plot_data))
x = plot_data["relative_iod_index"].to_numpy()
low = plot_data["relative_iod_ci_low"].to_numpy()
high = plot_data["relative_iod_ci_high"].to_numpy()
colors = np.where(plot_data["n_images"].eq(1), "#D17B29", "#2E6F95")
fig, ax = plt.subplots(figsize=(11, 11))
ax.hlines(y, low, high, color=colors, linewidth=2)
ax.scatter(x, y, color=colors, s=55, zorder=3)
ax.axvline(1.0, color="#555555", linestyle="--", linewidth=1)
ax.set_yticks(y, [short_species(value) for value in plot_data["species"]])
ax.set_xlabel("Relative nuclear-IOD index (median species = 1.0)")
ax.set_title("Equal-image species estimates with 95% hierarchical bootstrap intervals")
ax.grid(axis="x", alpha=0.18)
ax.text(
    0.01,
    0.01,
    "Orange = one observed image/specimen",
    transform=ax.transAxes,
    color="#9A541C",
)
save_figure(fig, "02_relative_iod_estimates.png")
plt.show()


## Image-to-image variation within species


In [ ]:
species_order = species_summary.sort_values("iod_equal_image_estimate")["species"].tolist()
fig, ax = plt.subplots(figsize=(12, 11))
for y, species in enumerate(species_order):
    image_values = image_summary.loc[
        image_summary["species"].eq(species), "image_median_iod"
    ].to_numpy()
    estimate = species_summary.loc[
        species_summary["species"].eq(species), "iod_equal_image_estimate"
    ].iloc[0]
    if len(image_values) > 1:
        ax.hlines(y, image_values.min(), image_values.max(), color="#9AA6B2", linewidth=2)
    ax.scatter(image_values, np.full(len(image_values), y), s=52, color="#365F78", alpha=0.85)
    ax.scatter(estimate, y, s=58, marker="D", color="#B33A3A", zorder=4)
ax.set_yticks(range(len(species_order)), [short_species(value) for value in species_order])
ax.set_xlabel("Image-specific median nuclear IOD")
ax.set_title("Observed image/specimen medians; red diamond = equal-image estimate")
ax.grid(axis="x", alpha=0.18)
save_figure(fig, "03_image_level_iod.png")
plt.show()
display(image_summary.sort_values(["species", "filename"]))


## What makes up IOD?

            Nuclear IOD is algebraically the nuclear mask area in pixels
            multiplied by mean optical density. Its association with nuclear
            area therefore cannot independently validate a biological
            genome-size mechanism. Both components are shown explicitly.


In [ ]:
component_data = species_summary.copy()
rho_area, p_area = spearmanr(
    component_data["median_nucleus_area_um2"],
    component_data["iod_equal_image_estimate"],
)
rho_od, p_od = spearmanr(
    component_data["median_nucleus_mean_od"],
    component_data["iod_equal_image_estimate"],
)
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
specs = [
    (
        "median_nucleus_area_um2",
        "Median nucleus area (µm²)",
        rho_area,
        p_area,
        "#477998",
    ),
    (
        "median_nucleus_mean_od",
        "Median nuclear mean optical density",
        rho_od,
        p_od,
        "#7A5195",
    ),
]
for ax, (column, xlabel, rho, p_value, color) in zip(axes, specs):
    ax.scatter(
        component_data[column],
        component_data["relative_iod_index"],
        s=48,
        color=color,
        alpha=0.85,
    )
    for row in component_data.itertuples():
        ax.annotate(
            short_species(row.species),
            (getattr(row, column), row.relative_iod_index),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=7,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Relative nuclear-IOD index")
    ax.set_title(f"Spearman ρ = {rho:.2f}; descriptive p = {p_value:.3g}")
    ax.grid(alpha=0.16)
fig.suptitle("Species IOD estimates against their two measurement components", y=1.02)
save_figure(fig, "04_iod_components.png")
plt.show()
print(f"IOD vs median nucleus area: rho={rho_area:.3f}, p={p_area:.4g}")
print(f"IOD vs median mean OD: rho={rho_od:.3f}, p={p_od:.4g}")


## Aggregation-method sensitivity


In [ ]:
sensitivity = species_summary[[
    "species",
    "iod_equal_image_estimate",
    "iod_pooled_nucleus_median",
    "iod_trimmed_mean_10pct",
]].copy()
method_columns = [
    "iod_equal_image_estimate",
    "iod_pooled_nucleus_median",
    "iod_trimmed_mean_10pct",
]
method_labels = ["Equal-image", "Pooled median", "10% trimmed mean"]
for column in method_columns:
    sensitivity[column + "_index"] = (
        sensitivity[column] / sensitivity[column].median()
    )
order = species_summary.sort_values("relative_iod_index")["species"].tolist()
fig, ax = plt.subplots(figsize=(12, 11))
offsets = [-0.18, 0.0, 0.18]
markers = ["D", "o", "s"]
colors = ["#B33A3A", "#365F78", "#6A994E"]
for y, species in enumerate(order):
    row = sensitivity.loc[sensitivity["species"].eq(species)].iloc[0]
    values = [row[column + "_index"] for column in method_columns]
    ax.plot(values, [y] * 3, color="#B8C0C8", linewidth=1, zorder=1)
    for offset, value, marker, color in zip(offsets, values, markers, colors):
        ax.scatter(value, y + offset, marker=marker, s=42, color=color, zorder=2)
ax.axvline(1.0, color="#555555", linestyle="--", linewidth=1)
ax.set_yticks(range(len(order)), [short_species(value) for value in order])
ax.set_xlabel("Within-method relative index (method median = 1.0)")
ax.set_title("Species estimates are shown under three transparent aggregation choices")
handles = [
    plt.Line2D([], [], marker=marker, linestyle="", color=color, label=label)
    for marker, color, label in zip(markers, colors, method_labels)
]
ax.legend(handles=handles, frameon=False, loc="lower right")
ax.grid(axis="x", alpha=0.18)
save_figure(fig, "05_aggregation_sensitivity.png")
plt.show()
display(
    species_summary[[
        "species",
        "pooled_median_difference_pct",
        "trimmed_mean_difference_pct",
    ]]
    .assign(max_abs_shift_pct=lambda x: x[[
        "pooled_median_difference_pct", "trimmed_mean_difference_pct"
    ]].abs().max(axis=1))
    .sort_values("max_abs_shift_pct", ascending=False)
)


## Technical-quality balance and residual IOD associations


In [ ]:
display(quality_balance)
display(quality_residual)

balance_plot = quality_balance.sort_values("species")
mean_columns = [
    "standardized_mean_difference_match_log_edge_sharpness",
    "standardized_mean_difference_match_log_relative_ring_noise",
]
matrix = balance_plot[mean_columns].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(15, 10), gridspec_kw={"width_ratios": [1.0, 1.25]})
image = axes[0].imshow(matrix, cmap="coolwarm", vmin=-0.10, vmax=0.10, aspect="auto")
axes[0].set_yticks(range(len(balance_plot)), [short_species(v) for v in balance_plot["species"]])
axes[0].set_xticks([0, 1], ["Edge sharpness", "Ring noise"], rotation=25, ha="right")
axes[0].set_title("Standardized mean difference vs reviewed template")
fig.colorbar(image, ax=axes[0], shrink=0.7, label="Standardized mean difference")

centered_log_iod = np.log(frozen["nuc_iod"]) - np.log(frozen["nuc_iod"]).groupby(
    frozen["species"]
).transform("median")
feature_colors = ["#477998", "#9C6644"]
for feature, color in zip(analysis.MATCH_FEATURES, feature_colors):
    centered_quality = frozen[feature] - frozen[feature].groupby(
        frozen["species"]
    ).transform("median")
    rho = quality_residual.loc[
        quality_residual["quality_feature"].eq(feature),
        "within_species_centered_spearman_rho",
    ].iloc[0]
    axes[1].scatter(
        centered_quality,
        centered_log_iod,
        s=13,
        alpha=0.30,
        color=color,
        label=f"{feature.replace('match_log_', '')}: ρ={rho:.2f}",
    )
axes[1].axhline(0, color="#777777", linewidth=0.8)
axes[1].axvline(0, color="#777777", linewidth=0.8)
axes[1].set_xlabel("Within-species centered quality feature")
axes[1].set_ylabel("Within-species centered log IOD")
axes[1].set_title("Residual technical-quality association")
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.14)
save_figure(fig, "06_quality_diagnostics.png")
plt.show()
print(
    "Worst absolute standardized mean difference:",
    round(quality_balance["max_abs_standardized_mean_difference"].max(), 3),
)
print(
    "Worst KS distance:",
    round(quality_balance["max_ks_distance"].max(), 3),
)


## Direct reading of the current results


In [ ]:
top = species_summary.iloc[0]
bottom = species_summary.iloc[-1]
fold = top["iod_equal_image_estimate"] / bottom["iod_equal_image_estimate"]
single_image = species_summary.loc[
    species_summary["n_images"].eq(1), "species"
].tolist()
intervals_crossing_anchor = int((
    species_summary["relative_iod_ci_low"].le(1.0)
    & species_summary["relative_iod_ci_high"].ge(1.0)
).sum())
method_shift = species_summary.assign(
    max_shift=lambda x: x[[
        "pooled_median_difference_pct", "trimmed_mean_difference_pct"
    ]].abs().max(axis=1)
).sort_values("max_shift", ascending=False).iloc[0]
rho_area, p_area = spearmanr(
    species_summary["median_nucleus_area_um2"],
    species_summary["iod_equal_image_estimate"],
)
rho_od, p_od = spearmanr(
    species_summary["median_nucleus_mean_od"],
    species_summary["iod_equal_image_estimate"],
)

display(HTML(f"""
<div style="border-left:5px solid #365F78;background:#f4f7f9;padding:14px 18px">
<p><b>Largest relative IOD:</b> {top['species']} ({top['relative_iod_index']:.3f}× the species-median anchor).</p>
<p><b>Smallest relative IOD:</b> {bottom['species']} ({bottom['relative_iod_index']:.3f}×).</p>
<p><b>Observed range:</b> {fold:.2f}-fold from highest to lowest.</p>
<p><b>Rank resolution:</b> {intervals_crossing_anchor} of {len(species_summary)}
intervals cross the 1.0 anchor, so most exact middle ranks should not be treated as
sharply separated.</p>
<p><b>Aggregation sensitivity:</b> the largest shift from equal-image weighting is
{method_shift['max_shift']:.1f}% for {method_shift['species']}.</p>
<p><b>IOD components:</b> species IOD correlates with median nucleus area
(ρ={rho_area:.2f}) and median mean optical density (ρ={rho_od:.2f}). This is expected
because both are algebraic components of IOD.</p>
<p><b>Single-image limitation:</b> {', '.join(single_image)} have only one observed
image/specimen, so their interval cannot measure between-image variation.</p>
</div>
"""))


## Conclusions and next validation step

            1. The frozen panel supports a stable **relative ranking** of
               nuclear IOD across the 20 image-quality-compatible species.
            2. Equal-image weighting is the primary estimate because it avoids
               letting images with more accepted nuclei dominate.
            3. The raw values, image-specific medians, bootstrap intervals,
               alternative aggregations, and technical-quality diagnostics
               remain visible rather than collapsing the evidence to one
               number per species.
            4. Most middle-ranked species have overlapping intervals. The point
               ranking is descriptive, not evidence that every adjacent pair
               differs biologically.
            5. This analysis cannot convert IOD to absolute genome size. That
               requires an independently measured DNA-content standard
               processed under the same staining and imaging protocol.
            6. *D. ochrophaeus* remains excluded from the primary comparison
               because its technical image-quality distribution had limited
               overlap; it belongs only in a labeled sensitivity analysis.


## Reproducibility and exported files


In [ ]:
exports = pd.DataFrame([
    {"artifact": "Frozen 721-nucleus source", "path": str(analysis.FROZEN_PATH)},
    {"artifact": "Species result table", "path": str(analysis.SPECIES_SUMMARY_PATH)},
    {"artifact": "Image result table", "path": str(analysis.IMAGE_SUMMARY_PATH)},
    {"artifact": "Frozen quality balance", "path": str(analysis.QUALITY_BALANCE_PATH)},
    {"artifact": "Residual quality diagnostics", "path": str(analysis.QUALITY_RESIDUAL_PATH)},
    {"artifact": "Analysis manifest", "path": str(analysis.ANALYSIS_MANIFEST_PATH)},
    {"artifact": "Executed notebook", "path": str(analysis.EXECUTED_NOTEBOOK_PATH)},
    {"artifact": "Rendered HTML", "path": str(analysis.HTML_PATH)},
])
display(exports)
print("Build source notebook:")
print("  uv run python path_analysis/scripts/build_frozen_genome_iod_notebook.py")
print("Execute notebook:")
print(
    "  uv run jupyter nbconvert --to notebook --execute "
    "path_analysis/notebooks/frozen_genome_iod_analysis.ipynb "
    "--output frozen_genome_iod_analysis.executed.ipynb "
    "--output-dir path_analysis/notebooks --ExecutePreprocessor.timeout=300"
)
print("Render HTML:")
print(
    "  uv run jupyter nbconvert --to html "
    "path_analysis/notebooks/frozen_genome_iod_analysis.executed.ipynb "
    "--output index.html "
    "--output-dir path_analysis/results/frozen_genome_iod_analysis"
)
